# Create a catalog & schema 

# With the help of autoloader, we will ingest data 


In [0]:
CATALOG="File_based"
SCHEMA="ecommerce"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.filelanding")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.checkpoints")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.schema")
          

In [0]:

CHECKPOINT_BASE = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"

File_landing_path = f"/Volumes/{CATALOG}/{SCHEMA}/filelanding"

Schema_inference_path = f"/Volumes/{CATALOG}/{SCHEMA}/schema"

In [0]:
files=dbutils.fs.ls(File_landing_path)
print(len(files))

for file in files:
    print(f" - {file.name}  ({file.size} bytes)")

In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_df=spark.readStream\
    .format("cloudFiles")\
    .option("cloudFiles.format","json")\
    .option("cloudFiles.schemaLocation",f"{Schema_inference_path}")\
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
    .option("cloudFiles.inferColumnTypes","true")\
    .option("cloudFiles.useNotifications","false")\
    .load(f"{File_landing_path}")\
    .withColumn('source_file_name', F.col("_metadata.file_name"))\
    .withColumn("file_name", F.col("_metadata.file_name"))\
    .withColumn("file_size", F.col("_metadata.file_size"))\
    .withColumn("modified_time", F.col("_metadata.file_modification_time"))\
    .withColumn("_ingest_time",F.current_timestamp())\
    .withColumn("_source_format", F.lit("json"))



In [0]:
bronze_df.writeStream.format("delta")\
         .outputMode("append")\
         .option("checkpointLocation", f"{CHECKPOINT_BASE}/bronze")\
         .trigger(availableNow=True)\
         .toTable(f"{CATALOG}.{SCHEMA}.bronze_layer")\
         .awaitTermination()    

print(f"Bronze load complete.")                  

In [0]:
bronze_count = spark.table(f"{CATALOG}.{SCHEMA}.bronze_layer").count()
display(spark.table(f"{CATALOG}.{SCHEMA}.bronze_layer").limit(5))
print(f"\nTotal rows in Bronze: {bronze_count}"/Volumes/file_based/ecommerce/filelanding)

In [0]:
from datetime import datetime

print(datetime.now())